# NumJa GPU Fallback POC — JCuda/cuBLAS on NVIDIA T4

Future-only runbook for `bench.jcudapoc.GemmBench`: the production-HAL CPU baseline versus isolated JCuda 12.6.0/cuBLAS Dgemm.

Before running: in Colab select **Runtime > Change runtime type > T4 GPU**, then run all cells. This notebook is intentionally unexecuted evidence source; it contains no credentials and does not modify the TornadoVM POC notebook. The default benchmark size is N=4096.

## Step 0 — T4 preflight

Fail before building if Colab did not attach a T4. The later setup installs CUDA 12.6 because JCublas2 requires the CUDA 12 `libcublas.so.12` SONAME.

In [1]:
%%bash
set -eu
nvidia-smi | head -8
nvidia-smi --query-gpu=name,driver_version --format=csv,noheader | tee /tmp/jcuda-device-preflight.log
grep -qi 'T4' /tmp/jcuda-device-preflight.log || { echo 'ERROR: Colab T4 GPU is required.' >&2; exit 1; }


Sun Sep 13 07:11:05 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
Tesla T4, 580.82.07


## Step 1 — Future-only setup and Linux build

This cell installs JDK 21, Maven 3.9.15, and CUDA 12.6, writes the per-shell environment file, clones the Phase 7 branch over HTTPS, and builds on Linux so Maven resolves Linux-native JCuda artifacts. It does not download or manually install JCuda JARs.

In [2]:
%%bash
set -Eeu
trap 'echo "ERROR: setup failed at line $LINENO: $BASH_COMMAND" >&2' ERR

if ! java -version 2>&1 | grep -q '"21'; then
  apt-get update
  DEBIAN_FRONTEND=noninteractive apt-get install -y openjdk-21-jdk
fi
export JAVA_HOME=$(dirname $(dirname $(readlink -f $(which java))))

MVN_DIR=/opt/maven-3.9.15
if [ ! -d "$MVN_DIR" ]; then
  wget https://archive.apache.org/dist/maven/maven-3/3.9.15/binaries/apache-maven-3.9.15-bin.tar.gz -O /tmp/mvn.tgz
  tar -xzf /tmp/mvn.tgz -C /opt/
  mv /opt/apache-maven-3.9.15 "$MVN_DIR"
fi
export PATH="$MVN_DIR/bin:$PATH"

CUDA12_LIB=/usr/local/cuda-12.6/lib64
if [ ! -e "$CUDA12_LIB/libcublas.so.12" ]; then
  wget https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64/cuda-keyring_1.1-1_all.deb -O /tmp/cuda-keyring.deb
  dpkg -i /tmp/cuda-keyring.deb
  apt-get update
  DEBIAN_FRONTEND=noninteractive apt-get install -y --no-install-recommends cuda-toolkit-12-6
fi
test -e "$CUDA12_LIB/libcublas.so.12" || { echo "ERROR: missing $CUDA12_LIB/libcublas.so.12" >&2; exit 1; }

cat > /tmp/poc-env.sh <<EOF
export JAVA_HOME=$JAVA_HOME
export PATH=$MVN_DIR/bin:\$PATH
export LD_LIBRARY_PATH=/usr/local/cuda-12.6/lib64:\${LD_LIBRARY_PATH:-}
EOF
source /tmp/poc-env.sh

REPO=/content/java_ml
BRANCH=gsd/phase-07-gpu-fallback-poc-via-jcuda-cublas
if [ -d "$REPO/.git" ]; then
  git -C "$REPO" fetch origin "$BRANCH"
  git -C "$REPO" checkout "$BRANCH"
  git -C "$REPO" reset --hard "origin/$BRANCH"
else
  git clone --branch "$BRANCH" --single-branch https://github.com/minhhhduc/jml.git "$REPO"
fi
cd "$REPO"
mvn -B -pl modules/numja -am install -DskipTests
mvn -B -pl bench/jcuda-poc -am package -DskipTests
test -f bench/jcuda-poc/target/jcuda-poc-jar.jar
echo 4096 > /tmp/bench-n.txt
echo '=== SETUP OK: JDK 21, Maven 3.9.15, CUDA 12.6, Linux JCuda build ==='

Selecting previously unselected package cuda-keyring.
(Reading database ... 126952 files and directories currently installed.)
Preparing to unpack /tmp/cuda-keyring.deb ...
Unpacking cuda-keyring (1.1-1) ...
Setting up cuda-keyring (1.1-1) ...
Get:1 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ InRelease [3,631 B]
Get:2 https://cli.github.com/packages stable InRelease [4,685 B]
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,578 B]
Get:4 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2404/x86_64  InRelease [1,578 B]
Get:5 http://security.ubuntu.com/ubuntu noble-security InRelease [126 kB]
Get:6 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ Packages [72.8 kB]
Get:7 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [3,007 kB]
Get:8 https://r2u.stat.illinois.edu/ubuntu noble InRelease [9,161 B]
Hit:9 http://archive.ubuntu.com/ubuntu noble InRelease
Get:10 https://dev

--2026-09-13 07:11:05--  https://archive.apache.org/dist/maven/maven-3/3.9.15/binaries/apache-maven-3.9.15-bin.tar.gz
Resolving archive.apache.org (archive.apache.org)... 65.108.204.189, 2a01:4f9:1a:a084::2
Connecting to archive.apache.org (archive.apache.org)|65.108.204.189|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 9236330 (8.8M) [application/x-gzip]
Saving to: ‘/tmp/mvn.tgz’

     0K .......... .......... .......... .......... ..........  0%  137K 65s
    50K .......... .......... .......... .......... ..........  1%  276K 49s
   100K .......... .......... .......... .......... ..........  1% 18.1M 32s
   150K .......... .......... .......... .......... ..........  2%  281K 32s
   200K .......... .......... .......... .......... ..........  2% 10.3M 26s
   250K .......... .......... .......... .......... ..........  3% 21.1M 21s
   300K .......... .......... .......... .......... ..........  3% 15.7M 18s
   350K .......... .......... .......... .......

## Step 2 — Benchmark and hard completion gate

`gpu_ms` is JCuda's kernel-only Dgemm timing (including device synchronization). `transfer_ms` is H2D plus D2H transfer time. Stderr is retained in `/tmp/poc-gpu.log` for native/device/number errors.

Two independent hard gates must both pass:

1. Java exits zero and the log contains exactly one `result=` line, `result=OK`.
2. In-process CUDA evidence: `cuda_alloc_delta_mib` (measured with `cudaMemGetInfo` inside the JVM around the three `cudaMalloc` calls) must be positive and at least half of `cuda_alloc_expected_mib`.

External `nvidia-smi` telemetry is printed for information only and never gates the run: some Colab containers report `[N/A]` for these fields, which is not evidence of an idle GPU.

Failures still preserve `/tmp/poc-gpu.log` as D-09 NO-GO evidence.

In [3]:
%%bash
set -eu
source /tmp/poc-env.sh
cd /content/java_ml
N=$(cat /tmp/bench-n.txt)
JAR=bench/jcuda-poc/target/jcuda-poc-jar.jar
test -e /usr/local/cuda-12.6/lib64/libcublas.so.12
test -f "$JAR"

rm -f /tmp/nv-sample.log /tmp/poc-gpu.log
# External telemetry is informational only: some Colab containers report [N/A] here.
nvidia-smi --query-gpu=timestamp,utilization.gpu,memory.used --format=csv,noheader,nounits -lms 100 -f /tmp/nv-sample.log 2>/dev/null &
NVPID=$!
trap 'kill "$NVPID" 2>/dev/null || true' EXIT
sleep 0.5
java -Dbench.env=colab -Dbench.size="$N" -jar "$JAR" 2>&1 | tee /tmp/poc-gpu.log
java_status=${PIPESTATUS[0]}
kill "$NVPID" 2>/dev/null || true
trap - EXIT
if [ "$java_status" -ne 0 ]; then
  echo "ERROR: GPU benchmark exited $java_status; preserve /tmp/poc-gpu.log as NO-GO evidence." >&2
  exit 1
fi
if [ "$(grep -c '^result=' /tmp/poc-gpu.log || true)" -ne 1 ] || ! grep -qx 'result=OK' /tmp/poc-gpu.log; then
  echo 'ERROR: GPU benchmark must emit exactly one result=OK; preserve /tmp/poc-gpu.log as NO-GO evidence.' >&2
  exit 1
fi

# Hard device-use evidence: in-process cudaMemGetInfo must show the 3 device buffers.
alloc_delta=$(grep -oP '^cuda_alloc_delta_mib=\K.*' /tmp/poc-gpu.log | tail -1 || true)
alloc_expected=$(grep -oP '^cuda_alloc_expected_mib=\K.*' /tmp/poc-gpu.log | tail -1 || true)
if [ -z "$alloc_delta" ]; then
  echo 'ERROR: missing cuda_alloc_delta_mib; preserve /tmp/poc-gpu.log as NO-GO evidence.' >&2
  exit 1
fi
if ! awk -v d="$alloc_delta" -v e="$alloc_expected" 'BEGIN { exit !((d + 0) > 0 && (d + 0) >= (e + 0) * 0.5) }'; then
  echo "ERROR: in-process CUDA allocation evidence too small (delta=${alloc_delta} MiB, expected>=${alloc_expected} MiB); preserve /tmp/poc-gpu.log as NO-GO evidence." >&2
  exit 1
fi
echo "OK: in-process CUDA allocation evidence delta=${alloc_delta} MiB (expected>=${alloc_expected} MiB)"

awk -F', ' '
$2 ~ /^[0-9]/ && $3 ~ /^[0-9]/ {
  n++; util += $2; mem += $3
  if ($2 > utilPeak) utilPeak = $2
  if ($3 > memPeak) memPeak = $3
}
END {
  print "gpu_telemetry_samples=" n + 0
  printf "gpu_util_peak_pct=%.0f", utilPeak + 0; print ""
  printf "gpu_mem_peak_mb=%.0f", memPeak + 0; print ""
  if (n == 0) print "note=external nvidia-smi telemetry unavailable or [N/A]; in-process CUDA evidence is authoritative"
}' /tmp/nv-sample.log

[0.000s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
cpu_sample_1_ms=6.725854e+04
cpu_sample_2_ms=6.856316e+04
cpu_sample_3_ms=6.734038e+04
gpu_sample_1_ms=5.487216e+02
gpu_sample_2_ms=5.484919e+02
gpu_sample_3_ms=5.485834e+02
cuda_device_index=0
cuda_total_mib=1.491269e+04
cuda_free_mib_before_alloc=1.480781e+04
cuda_free_mib_after_alloc=1.442381e+04
cuda_alloc_delta_mib=3.840000e+02
cuda_alloc_expected_mib=3.840000e+02
cpu_vs_gpu_max_abs_err=9.094947e-13
cpu_vs_gpu_max_rel_err=8.881206e-16
cpu_vs_gpu_mae=5.426763e-14
cpu_baseline_ms=67340.383673
gpu_ms=548.583396
transfer_ms=95.084199
speedup_ratio=122.75322979881076
transfer_pct=17.332678986149993
device=Tesla T4
result=OK
verdict=GO
env=colab
jdk=21.0.12
hardwa

## Step 3 — NumPy magnitude reference

This is a magnitude-only reference. Java `Random` and NumPy's generator produce different matrices, so `GemmBench`'s full-matrix Frobenius error is the authoritative accuracy check.

In [4]:
import numpy as np
N = int(open('/tmp/bench-n.txt').read().strip())
a_np = np.random.default_rng(0xC0FFEE).random((N, N))
b_np = np.random.default_rng(0xBADF00D).random((N, N))
c_np = a_np @ b_np
print(f'N={N}')
print(f'||C_np||_F = {np.linalg.norm(c_np, "fro"):.6e}')
print(f'max |C_np| = {np.abs(c_np).max():.6e}')
print('Magnitude reference only; GemmBench Frobenius output is authoritative.')


N=4096
||C_np||_F = 4.194407e+06
max |C_np| = 1.094033e+03
Magnitude reference only; GemmBench Frobenius output is authoritative.


## Step 4 — Decision summary and rubric

Copy the retained log, environment metadata, telemetry, and summary into `07-RESULTS.md`. Numerical gate: `cpu_vs_gpu_frob_rel_err <= 1e-9`. Performance GO requires both `speedup_ratio >= 2.0` and `transfer_pct < 50.0`; otherwise record NO-GO. Do not try a third GPU backend if this POC fails.

In [5]:
%%bash
set -eu
echo '=== CPU vs GPU timing samples (GPU kernel-only; transfer excluded) ==='
grep -E '^(cpu_sample_[1-3]_ms|gpu_sample_[1-3]_ms|cpu_baseline_ms|gpu_ms|transfer_ms|speedup_ratio|transfer_pct)=' /tmp/poc-gpu.log
echo '=== In-process CUDA device memory evidence ==='
grep -E '^cuda_(device_index|total_mib|free_mib_before_alloc|free_mib_after_alloc|alloc_delta_mib|alloc_expected_mib)=' /tmp/poc-gpu.log
echo '=== Result ==='
grep -E '^(env|device|jdk|hardware|size|cpu_vs_gpu_(frob_rel_err|max_abs_err|max_rel_err|mae)|result|verdict)=' /tmp/poc-gpu.log
echo '=== External telemetry (informational) ==='
awk -F', ' '
$2 ~ /^[0-9]/ && $3 ~ /^[0-9]/ {
  n++; util += $2; mem += $3
  if ($2 > utilPeak) utilPeak = $2
  if ($3 > memPeak) memPeak = $3
}
END {
  print "gpu_telemetry_samples=" n + 0
  printf "gpu_util_peak_pct=%.0f", utilPeak + 0; print ""
  printf "gpu_mem_peak_mb=%.0f", memPeak + 0; print ""
  if (n == 0) print "note=external nvidia-smi telemetry unavailable or [N/A]; in-process CUDA evidence is authoritative"
}' /tmp/nv-sample.log
verdict=$(grep -oP '^verdict=\K.*' /tmp/poc-gpu.log | tail -1 || true)
frob=$(grep -oP '^cpu_vs_gpu_frob_rel_err=\K.*' /tmp/poc-gpu.log | tail -1 || true)
speedup=$(grep -oP '^speedup_ratio=\K.*' /tmp/poc-gpu.log | tail -1 || true)
transfer=$(grep -oP '^transfer_pct=\K.*' /tmp/poc-gpu.log | tail -1 || true)
printf 'Auto-verdict: %s\nFrobenius: %s\nSpeedup: %s\nTransfer: %s\n' "${verdict:-?}" "${frob:-?}" "${speedup:-?}" "${transfer:-?}"
case "$verdict" in GO) echo 'Recommendation: GO — consider later HAL integration.' ;; NO-GO) echo 'Recommendation: NO-GO — retain evidence and defer GPU.' ;; *) echo 'INSUFFICIENT_DATA — inspect /tmp/poc-gpu.log.' ;; esac

=== CPU vs GPU timing samples (GPU kernel-only; transfer excluded) ===
cpu_sample_1_ms=6.725854e+04
cpu_sample_2_ms=6.856316e+04
cpu_sample_3_ms=6.734038e+04
gpu_sample_1_ms=5.487216e+02
gpu_sample_2_ms=5.484919e+02
gpu_sample_3_ms=5.485834e+02
cpu_baseline_ms=67340.383673
gpu_ms=548.583396
transfer_ms=95.084199
speedup_ratio=122.75322979881076
transfer_pct=17.332678986149993
=== In-process CUDA device memory evidence ===
cuda_device_index=0
cuda_total_mib=1.491269e+04
cuda_free_mib_before_alloc=1.480781e+04
cuda_free_mib_after_alloc=1.442381e+04
cuda_alloc_delta_mib=3.840000e+02
cuda_alloc_expected_mib=3.840000e+02
=== Result ===
cpu_vs_gpu_max_abs_err=9.094947e-13
cpu_vs_gpu_max_rel_err=8.881206e-16
cpu_vs_gpu_mae=5.426763e-14
device=Tesla T4
result=OK
verdict=GO
env=colab
jdk=21.0.12
hardware=Linux+amd64+2cores
size=4096
cpu_vs_gpu_frob_rel_err=9.848062e-17
=== External telemetry (informational) ===
gpu_telemetry_samples=2700
gpu_util_peak_pct=100
gpu_mem_peak_mb=501
Auto-verdict: G